# 🧬 Single-Cell RNA-seq Analysis with Scanpy

## Learning Objectives
- Understand single-cell RNA-seq data structures (AnnData)
- Perform quality control and preprocessing
- Apply dimensionality reduction (PCA, UMAP)
- Identify cell clusters and marker genes

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, frameon=False)

## 1. The AnnData Structure

Single-cell data uses the **AnnData** format:
- `X`: Expression matrix (cells × genes)
- `obs`: Cell metadata (DataFrame)
- `var`: Gene metadata (DataFrame)
- `obsm`: Embeddings (PCA, UMAP)
- `uns`: Unstructured data

In [ ]:
# Load example dataset (PBMC 3k from 10X Genomics)
adata = sc.datasets.pbmc3k()
print(f"Dataset shape: {adata.shape[0]} cells × {adata.shape[1]} genes")
adata

## 2. Quality Control

Filter cells based on:
- Number of genes detected
- Total counts (library size)
- Mitochondrial gene percentage (cell stress indicator)

In [ ]:
# Mark mitochondrial genes
adata.var['mt'] = adata.var_names.str.startswith('MT-')

# Calculate QC metrics
sc.pp.calculate_qc_metrics(
    adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True
)

# Visualize QC metrics
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
sc.pl.violin(adata, 'n_genes_by_counts', ax=axes[0], show=False)
sc.pl.violin(adata, 'total_counts', ax=axes[1], show=False)
sc.pl.violin(adata, 'pct_counts_mt', ax=axes[2], show=False)
plt.tight_layout()
plt.show()

In [ ]:
# Filter cells
print(f"Before filtering: {adata.n_obs} cells")

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[adata.obs.pct_counts_mt < 20, :]

print(f"After filtering: {adata.n_obs} cells")

## 3. Normalization and Highly Variable Genes

In [ ]:
# Normalize to 10,000 counts per cell
sc.pp.normalize_total(adata, target_sum=1e4)

# Log transform
sc.pp.log1p(adata)

# Find highly variable genes (for dimensionality reduction)
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
print(f"Highly variable genes: {sum(adata.var.highly_variable)}")

# Plot
sc.pl.highly_variable_genes(adata)

## 4. Dimensionality Reduction

In [ ]:
# Save raw counts for differential expression later
adata.raw = adata

# Subset to highly variable genes
adata = adata[:, adata.var.highly_variable]

# Scale data
sc.pp.scale(adata, max_value=10)

# PCA
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, n_pcs=50)

In [ ]:
# Compute neighborhood graph
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

# UMAP embedding
sc.tl.umap(adata)
sc.pl.umap(adata, color=['n_genes_by_counts', 'total_counts', 'pct_counts_mt'])

## 5. Clustering

In [ ]:
# Leiden clustering
sc.tl.leiden(adata, resolution=0.5)

# Visualize clusters
sc.pl.umap(adata, color='leiden', legend_loc='on data')

## 6. Marker Gene Detection

In [ ]:
# Find marker genes
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

In [ ]:
# Check known markers
marker_genes = {
    'T cells': ['CD3D', 'CD3E', 'CD4', 'CD8A'],
    'B cells': ['CD19', 'MS4A1', 'CD79A'],
    'Monocytes': ['CD14', 'LYZ', 'FCGR3A'],
    'NK cells': ['GNLY', 'NKG7']
}

# Flatten marker list
all_markers = [m for markers in marker_genes.values() for m in markers]
available = [m for m in all_markers if m in adata.raw.var_names]

sc.pl.dotplot(adata, available, groupby='leiden', standard_scale='var')

## 🧬 Exercise: Annotate Cell Types

Based on the marker gene expression, assign cell type labels to each cluster.

In [ ]:
# Your code here
# cluster_annotations = {
#     '0': 'T cells',
#     '1': 'Monocytes',
#     # ...
# }
# adata.obs['cell_type'] = adata.obs['leiden'].map(cluster_annotations)

## Summary

**Scanpy workflow:**
1. Load data → AnnData
2. QC filtering (genes, counts, MT%)
3. Normalize and log-transform
4. Find highly variable genes
5. PCA → Neighborhood graph → UMAP
6. Cluster (Leiden)
7. Find markers → Annotate cell types